# 06 · Traditional OCR baselines

Run two OCR + rules pipelines on the same test set, schema and metrics as the three
VLMs, and write the reports into `result/{side}/` for notebook 05 to merge.

| Pipeline | Detection | Recognition | Role |
|---|---|---|---|
| `paddleocr` | PP-OCR DB | PP-OCR CRNN (`lang=vi`) | generic multilingual OCR |
| `paddle_vietocr` | PP-OCR DB | VietOCR `vgg_transformer` | Vietnamese-specific OCR |

Both pipelines share a detector on purpose: holding line detection fixed makes the gap
between them attributable to recognition alone.

This notebook closes the limitation that the work had not shown VLM + QLoRA beating an
OCR + rules pipeline on this dataset.

---

> ### ⚠️ Run this in a separate Colab session
>
> Installing `paddleocr` tends to change `numpy`/`opencv` versions and break the working
> `torch`/`transformers` of the VLM branch. If you must share a session, install the
> baseline **last** and re-check with `import torch, transformers`.

**Prerequisite:** notebook 02 produced `test.jsonl` at
`{DATA_DRIVE}/dataset/{SIDE}/test.jsonl` (not inside the repo). Images live in
`{DATA_DRIVE}/{SIDE}/`.

**Runtime:** a GPU is optional but helpful — see section 2b for which half benefits.

> **The Drive folder must be under "My Drive".** Colab mounts My Drive and Workspace
> shared drives only; *"Shared with me"* has no mount path. If `cccd_project` lives
> there, right-click → *Organise* → *Add shortcut to Drive* → **My Drive**, then remount.

## 1. Mount Drive and set paths

In [2]:
SIDE    = 'Back'                              # must match notebooks 03 and 05
ENGINES = ['paddleocr', 'paddle_vietocr']

from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

DATA_DRIVE = Path('/content/drive/MyDrive/cccd_project/Data')
REPO_DRIVE = DATA_DRIVE / 'label_CCCD'

# Notebook 05 reads from here
RESULT_DIR = DATA_DRIVE / 'result' / SIDE.lower()
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TEST_JSONL = DATA_DRIVE / 'dataset' / SIDE / 'test.jsonl'
IMAGE_ROOT = DATA_DRIVE / SIDE

sys.path.insert(0, str(REPO_DRIVE))
%cd {REPO_DRIVE}

print('Repo      :', REPO_DRIVE)
print('Ket qua   :', RESULT_DIR)
print('Test set  :', TEST_JSONL)
print('           ', 'OK' if TEST_JSONL.exists() else '>>> KHONG THAY - chay notebook 02 truoc')

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1WXAxixNRVv8WOValCqogSZbQJBkxP0Zv/cccd_project/Data/label_CCCD
Repo      : /content/drive/MyDrive/cccd_project/Data/label_CCCD
Ket qua   : /content/drive/MyDrive/cccd_project/Data/result/back
Test set  : /content/drive/MyDrive/cccd_project/Data/dataset/Back/test.jsonl
            OK


In [3]:
# Check test.jsonl holds the expected side
import json
from src.utils.cccd_schema import infer_side_from_filename

records = [json.loads(l) for l in TEST_JSONL.read_text(encoding='utf-8').splitlines() if l.strip()]
sides = {}
for r in records:
    s = infer_side_from_filename(r['image'])
    sides[getattr(s, 'value', str(s))] = sides.get(getattr(s, 'value', str(s)), 0) + 1

print(f'{len(records)} record trong test.jsonl')
print('Phan bo mat the:', sides)
if len(sides) > 1:
    print()
    print('LUU Y: test.jsonl chua CA HAI mat. Baseline van chay duoc (parser tu suy mat')
    print('theo ten file), nhung report se tron 2 mat trong khi notebook 05 gom theo')
    print('SIDE. Neu notebook 03 chay rieng tung mat thi o day cung phai tro vao dung')
    print('file test cua mat do.')

161 record trong test.jsonl
Phan bo mat the: {'sau': 161}


## 2. Install dependencies

`requirements-ocr.txt` is deliberately separate from the VLM `requirements.txt` — see the
conflict warning at the top. The first VietOCR run downloads ~100MB of weights.

**Two cells, which cannot be merged.** `vietocr` must be installed with `--no-deps`
because it hard-pins `pillow==10.2.0`, which has no wheel for Colab's Python 3.13; the
flag is per command and cannot be declared inside a requirements file.

> The `paddleocr` engine only needs the first cell. The second is for `paddle_vietocr`.

In [4]:
!pip install -q -r "$REPO_DRIVE/requirements-ocr.txt"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.

In [5]:
# vietocr pins pillow==10.2.0, so skip its deps
!pip install -q --no-deps "vietocr>=0.3.11"
!pip install -q einops gdown prefetch-generator lmdb pyyaml

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vietocr 0.3.13 requires imgaug==0.4.0, which is not installed.
vietocr 0.3.13 requires albumentations==1.4.2, but you have albumentations 2.0.8 which is incompatible.
vietocr 0.3.13 requires einops==0.2.0, but you have einops 0.8.2 which is incompatible.
vietocr 0.3.13 requires gdown==4.4.0, but you have gdown 5.2.2 which is incompatible.
vietocr 0.3.13 requires pillow==10.2.0, but you have pillow 11.3.0 which is incompatible.
vietocr 0.3.13 requires prefetch-generator==1.0.1, but you have prefetch-generator 1.0.3 which is incompatible.


### 2c. Cache the VietOCR config and weights (required)

VietOCR downloads both the config YAML and the `.pth` weights from `vocr.vn`, whose TLS
certificate has expired, so `Cfg.load_config_from_name()` fails with:

```
SSLCertVerificationError: certificate verify failed: certificate has expired
```

This is a server-side problem and `REQUESTS_CA_BUNDLE` does not help — an expired
certificate is rejected by any CA bundle.

The cell below downloads three files once and caches them on Drive:

| File | Source | Certificate |
|---|---|---|
| `base.yml` | GitHub `pbcquoc/vietocr` | ✅ valid |
| `vgg-transformer.yml` | GitHub `pbcquoc/vietocr` | ✅ valid |
| `vgg_transformer.pth` (~150MB) | `vocr.vn` — no mirror | ⚠ needs `--no-check-certificate` |

Only one file skips certificate verification, and only for that command. After caching,
later runs never touch the network.

In [6]:
import os

VIETOCR_CACHE = DATA_DRIVE / 'vietocr_cache'
VIETOCR_CACHE.mkdir(parents=True, exist_ok=True)

# Configs from GitHub (valid certificate)
GH = 'https://raw.githubusercontent.com/pbcquoc/vietocr/master/config'
for name in ('base.yml', 'vgg-transformer.yml'):
    dst = VIETOCR_CACHE / name
    if not dst.exists():
        !wget -q -O '{dst}' '{GH}/{name}'
    print(f'{name:24s} {dst.stat().st_size:>8,} bytes')

# Weights only exist on vocr.vn (expired certificate)
W = VIETOCR_CACHE / 'vgg_transformer.pth'
if not W.exists():
    !wget --no-check-certificate -q --show-progress \
        -O '{W}' https://vocr.vn/data/vietocr/vgg_transformer.pth
print(f'{W.name:24s} {W.stat().st_size / 1024**2:>8.1f} MB')

# A failed wget still creates an empty file
for p in (VIETOCR_CACHE / 'base.yml', VIETOCR_CACHE / 'vgg-transformer.yml', W):
    if p.stat().st_size == 0:
        raise SystemExit(f'{p.name} tai ve RONG - xoa di roi chay lai cell nay.')

# Read by ocr_engines.load_vietocr_config()
os.environ['VIETOCR_CACHE_DIR'] = str(VIETOCR_CACHE)
print('\nVIETOCR_CACHE_DIR =', os.environ['VIETOCR_CACHE_DIR'])

base.yml                    1,809 bytes
vgg-transformer.yml           505 bytes
vgg_transformer.pth         144.8 MB

VIETOCR_CACHE_DIR = /content/drive/MyDrive/cccd_project/Data/vietocr_cache


## 2b. How far does the GPU help?

The two halves run on different runtimes:

- **VietOCR (recognition)** runs on `torch` and uses the GPU automatically. This is the
  expensive half and the one that benefits most.
- **PaddleOCR (detection, and recognition in the `paddleocr` engine)** needs a
  `paddlepaddle-gpu` build matching CUDA. `requirements-ocr.txt` installs the CPU build.

So the Colab default is: recognition on GPU, detection on CPU. The cell below prints what
is actually available.

In [6]:
print('--- VietOCR (recognition, chay tren torch) ---')
try:
    import torch
    print('  torch                :', torch.__version__)
    print('  CUDA kha dung        :', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('  GPU                  :', torch.cuda.get_device_name(0))
        print('  cuDNN                :', torch.backends.cudnn.version())
        # Real conv test: is_available() only checks the driver
        try:
            import torch.nn as nn
            _x = torch.randn(2, 3, 64, 64, device='cuda')
            nn.Conv2d(3, 8, 3).cuda()(_x)
            print('  conv qua cuDNN       : OK')
            print('  => VietOCR TU DONG dung cuda:0. Khong can truyen --device.')
        except Exception as exc:
            print('  conv qua cuDNN       : HONG ->', type(exc).__name__, exc)
            print('  >>> torch da bi mot lan pip install khac lam hong.')
            print('      Doc lai canh bao "dependency conflicts" cua pip: no ghi ro')
            print('      torch can nvidia-* ban nao. Cai dung ban do roi')
            print('      Runtime -> Restart session (thu vien CUDA da nap khong doi nong duoc).')
    else:
        print('  => Khong co GPU: VietOCR chay CPU, se cham hon nhieu.')
except ImportError:
    print('  chua cai torch')

print()
print('--- PaddleOCR (detection; va ca recognition o engine paddleocr) ---')
try:
    import paddle
    compiled = paddle.device.is_compiled_with_cuda()
    print('  paddle               :', paddle.__version__)
    print('  build co CUDA        :', compiled)
    print('  device mac dinh      :', paddle.device.get_device())
    if compiled:
        print('  => Co the them --gpu cho eval_baseline.py.')
    else:
        print('  => Ban CPU. Co --gpu se bi BO QUA kem canh bao trong log')
        print('     ("khong nhan kwarg nao mang device"). Muon GPU: xem cell duoi.')
except ImportError:
    print('  chua cai paddle')

print()
print('--- oneDNN (backend CPU cua Paddle) ---')
from src.baselines.ocr_engines import DISABLE_MKLDNN
print('  Se tat oneDNN        :', DISABLE_MKLDNN)
print('  => Paddle 3.x chay IR moi (PIR), ma nhanh oneDNN cua executor moi chua cai')
print('     dat ConvertPirAttribute2RuntimeAttribute cho ArrayAttribute<Double> =>')
print('     predictor.run() chet ngay lan detect dau tien. ocr_engines tu truyen')
print('     enable_mkldnn=False de tranh. Chi mat toc do CPU, KHONG doi FA/CER.')
print('     Bat lai bang bien moi truong PADDLE_ENABLE_MKLDNN=1 neu may khong dinh bug.')

--- VietOCR (recognition, chay tren torch) ---
  torch                : 2.11.0+cu128
  CUDA kha dung        : True
  GPU                  : Tesla T4
  cuDNN                : 91900
  conv qua cuDNN       : OK
  => VietOCR TU DONG dung cuda:0. Khong can truyen --device.

--- PaddleOCR (detection; va ca recognition o engine paddleocr) ---


/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


  paddle               : 3.3.1
  build co CUDA        : False
  device mac dinh      : cpu
  => Ban CPU. Co --gpu se bi BO QUA kem canh bao trong log
     ("khong nhan kwarg nao mang device"). Muon GPU: xem cell duoi.

--- oneDNN (backend CPU cua Paddle) ---
  Se tat oneDNN        : True
  => Paddle 3.x chay IR moi (PIR), ma nhanh oneDNN cua executor moi chua cai
     dat ConvertPirAttribute2RuntimeAttribute cho ArrayAttribute<Double> =>
     predictor.run() chet ngay lan detect dau tien. ocr_engines tu truyen
     enable_mkldnn=False de tranh. Chi mat toc do CPU, KHONG doi FA/CER.
     Bat lai bang bien moi truong PADDLE_ENABLE_MKLDNN=1 neu may khong dinh bug.


### (Optional) Install `paddlepaddle-gpu` for the detection half

> ### ⛔ Read this before running — this cell has broken `torch` before
>
> The Paddle GPU build pulls its own pinned `nvidia-*` packages and **downgrades** the
> ones `torch` uses (`nvidia-cudnn-cu12`, `nvidia-nccl-cu12`, `nvidia-cusparselt-cu12`).
> It breaks silently: `torch.cuda.is_available()` stays `True` and only cuDNN convolution
> fails — and VietOCR runs on torch. The conv test in section 2b catches exactly this.
>
> The payoff is small: the expensive half is VietOCR, which already uses the GPU through
> torch. Paddle GPU only speeds up detection, and `--gpu` does not affect FA/CER.

Two further obstacles:

- PyPI only has GPU builds up to **2.6.2** (cp312 max). On Colab's Python 3.13,
  `pip install paddlepaddle-gpu` reports `from versions: none`; 3.x lives only on Baidu's
  own index.
- `-i` **replaces** the index rather than adding one, so the Paddle GPU install must be a
  separate command or `paddleocr`/`vietocr` fail with it.

Look up the right command for the runtime's CUDA at
[paddlepaddle.org.cn/install/quick](https://www.paddlepaddle.org.cn/install/quick). After
installing, **Runtime → Restart session**, re-run from section 1, and re-run section 2b
to confirm `conv qua cuDNN : OK`.

In [ ]:
# OPTIONAL - read the warning above first
# Match cu126 to the runtime CUDA (!nvidia-smi)
#
# !pip uninstall -y paddlepaddle paddlepaddle-gpu
# !pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# Recovery if torch broke: restart the session after this
#
# !pip uninstall -y paddlepaddle-gpu
# !pip install -q "paddlepaddle>=2.5.0"
# !pip install -q nvidia-cudnn-cu12==9.19.0.56 nvidia-cusparselt-cu12==0.7.1 nvidia-nccl-cu12==2.28.9

## 3. Parser smoke test

Runs `field_parser` on mock layouts and compares against real reference labels. Needs no
paddle/vietocr and takes seconds on CPU.

This is the text → field mapping layer, which decides the baseline score and is the part
most likely to break when the line-join rules change. If this fails, do not spend time
running OCR over the whole set.

In [ ]:
!python -m src.baselines.field_parser


=== MẶT TRƯỚC ===
  ✓ so_cccd: '089203011696'
  ✓ ho_va_ten: 'NGUYỄN THƯỜNG QUÂN'
  ✓ ngay_sinh: '25/11/2003'
  ✓ gioi_tinh: 'Nam'
  ✓ quoc_tich: 'Việt Nam'
  ✓ que_quan: 'Phước Hưng, An Phú, An Giang'
  ✓ noi_thuong_tru: 'Ấp Phước Hòa, Phước Hưng, An Phú, An Giang'
  ✓ co_gia_tri_den: '25/11/2028'
  ✓ mat_the: 'truoc'

=== MẶT SAU ===
  ✓ dac_diem_nhan_dang: 'Sẹo chấm C:2cm trên sau cánh mũi trái'
  ✓ ngay_cap: '22/09/2021'
  ✓ noi_cap: 'CỤC TRƯỞNG CỤC CẢNH SÁT QUẢN LÝ HÀNH CHÍNH VỀ TRẬT TỰ XÃ HỘI'
  ✓ mat_the: 'sau'

✅ Tất cả trường khớp



## 4. Probe a single image first

Do not skip this. It prints every OCR line, the JSON the parser extracted and a per-field
diff against the labels — showing within 30 seconds whether OCR is misreading or the
field splitting is off.

In [ ]:
PROBE_IMAGE = IMAGE_ROOT / records[0]['image']
print('Anh probe:', PROBE_IMAGE)
print('Ton tai  :', PROBE_IMAGE.exists())

Anh probe: /content/drive/MyDrive/cccd_project/Data/Back/cccd_back_813.png
Ton tai  : True


In [ ]:
!python -m src.baselines.ocr_registry \
    --engine paddle_vietocr \
    --image '{PROBE_IMAGE}' \
    --gold '{TEST_JSONL}' \
    --show_rows


=== paddle_vietocr — PP-OCR det + VietOCR rec (vgg_transformer) ===
available   : True (ok)
detector    : ppocr_db
recognizer  : vietocr
2026-08-30 18:51:24,210 [INFO] 🔧 det_limit_side_len = 1024
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
2026-08-30 18:51:41,839 [INFO] PP-OCR TextDetection: khởi tạo với {'limit_side_len': 1024, 'device': 'cpu', 'enable_mkldnn': False}
2026-08-30 18:51:41,924 [INFO] VietOCR config đọc từ cache local: /content/drive/MyDrive/cccd_project/Data/vietocr_cache
2026-08-30 18:51:41,925 [INFO] VietOCR trọng số đọc từ cache local: /cont

## 5. Evaluate the whole test set

Everything that affects a control variable stays at its default:

- `--image_max_side` = 1024, the same `preprocess_image` the VLMs use.
- `--det_limit_side_len` left empty so it follows `--image_max_side`; otherwise PP-OCR
  silently shrinks to 960px before detection.
- `--canonicalize` **off** — enabling it is an ablation, not a headline number.

Each engine writes three files into `result/{side}/`: `eval_<engine>_<side>.json`,
`..._preds.jsonl` (needed by notebook 05) and `..._ocr.jsonl` (the OCR cache for
section 6).

> Timing: VietOCR on GPU takes minutes; everything on CPU can take tens of minutes. Add
> `--limit 20` for a quick trial run.

In [10]:
import time

for eng in ENGINES:
    rp = RESULT_DIR / f'eval_{eng}_{SIDE.lower()}.json'
    print()
    print('=' * 74)
    print(f'>>> {eng}  ->  {rp.name}')
    print('=' * 74)
    t0 = time.time()
    !python scripts/eval_baseline.py \
        --test_jsonl '{TEST_JSONL}' \
        --image_root '{IMAGE_ROOT}' \
        --engine {eng} \
        --report_path '{rp}'
    print(f'--- {eng}: {time.time() - t0:.0f}s')


>>> paddleocr  ->  eval_paddleocr_back.json
2026-09-02 04:48:57,001 [INFO] 🔄 Đang dựng engine paddleocr (161/161 ảnh chưa có cache)...
2026-09-02 04:48:57,002 [INFO] 🔧 det_limit_side_len = 1024
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
2026-09-02 04:49:24,304 [INFO] HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-09-02 04:49:24,544 [INFO] HTTP Request: GET https://huggingface.co/api/models/PaddlePaddle/PP-LCNet_x1_0_doc_ori/revision/main "HTTP/1.1 200 OK"
2026-09-02 04:49:24,791 [INFO] HTTP Request: GET https://huggingface.co/api/models/PaddlePaddle/PP-LCNet_x1_0_doc_ori/tree/d3b95a6dff5fe8a94f2748e12b61cb26818a0df8?recursive=true&expa

## 6. Re-score after editing the parser, without re-running OCR

`..._ocr.jsonl` stores every raw OCR line, so after editing
`src/baselines/field_parser.py` only the rules layer has to run: seconds on CPU instead
of tens of minutes of OCR.

The loop is: edit `field_parser.py` → run section 3 (smoke test) → run the cell below →
check FA.

In [ ]:
ENGINE_TO_RESCORE = 'paddle_vietocr'          # or 'paddleocr'

rp    = RESULT_DIR / f'eval_{ENGINE_TO_RESCORE}_{SIDE.lower()}.json'
cache = rp.with_name(rp.stem + '_ocr.jsonl')
print('Cache OCR:', cache.name, '-', 'OK' if cache.exists() else 'CHUA CO (chay muc 5 truoc)')
print()

!python scripts/eval_baseline.py \
    --test_jsonl '{TEST_JSONL}' \
    --image_root '{IMAGE_ROOT}' \
    --engine {ENGINE_TO_RESCORE} \
    --ocr_cache '{cache}' \
    --report_path '{rp}'

Cache OCR: eval_paddle_vietocr_back_ocr.jsonl - OK

2026-08-30 20:49:35,339 [INFO] 📂 Đã nạp cache OCR cho 161 ảnh từ /content/drive/MyDrive/cccd_project/Data/result/back/eval_paddle_vietocr_back_ocr.jsonl
2026-08-30 20:49:35,340 [INFO] 🚀 Bắt đầu chấm điểm baseline paddle_vietocr trên tập Test...
Đánh giá: 100% 161/161 [00:05<00:00, 29.42ảnh/s]
2026-08-30 20:49:40,840 [INFO] 💾 Đã ghi 161 dự đoán → /content/drive/MyDrive/cccd_project/Data/result/back/eval_paddle_vietocr_back_preds.jsonl
2026-08-30 20:49:40,992 [INFO] FA=0.5714 | CER=0.1092 | F1=0.8193 | fill=97.7% | 15.5 dòng/ảnh | p50=33.9ms
2026-08-30 20:49:40,992 [INFO] ✅ Đã lưu báo cáo → /content/drive/MyDrive/cccd_project/Data/result/back/eval_paddle_vietocr_back.json


## 7. Ablations (optional) — not headline numbers

These two flags answer whether the gap to the VLMs comes from misreading or from
presentation conventions and resolution:

| Flag | Measures | Why it is off by default |
|---|---|---|
| `--canonicalize` | the presentation-convention gap | forcing `noi_cap`/`quoc_tich` to a canonical sentence guesses from the label distribution, not from reading |
| `--image_max_side 0` | the resolution gap | gives the baseline an advantage none of the VLMs have |

Results go into the `ablation/` subdirectory, not `result/{side}/`. That matters:
`compare_models.py` groups by `model_key × mode`, and an ablation carries the same key
pair as the headline run (`paddle_vietocr` × `rule_based`), so sharing a directory would
let it silently overwrite the headline numbers.

In [7]:
ABL_DIR = RESULT_DIR / 'ablation'
ABL_DIR.mkdir(parents=True, exist_ok=True)

ENGINE_ABL = 'paddle_vietocr'
cache = RESULT_DIR / f'eval_{ENGINE_ABL}_{SIDE.lower()}_ocr.jsonl'

# Rules-layer only, so reuse the OCR cache
rp_canon = ABL_DIR / f'eval_{ENGINE_ABL}_{SIDE.lower()}_canon.json'
print('=' * 74); print('>>> ablation: canonicalize (tu cache OCR)'); print('=' * 74)
!python scripts/eval_baseline.py \
    --test_jsonl '{TEST_JSONL}' --image_root '{IMAGE_ROOT}' \
    --engine {ENGINE_ABL} --ocr_cache '{cache}' --canonicalize \
    --report_path '{rp_canon}'

# Different input images, so OCR must run again
rp_full = ABL_DIR / f'eval_{ENGINE_ABL}_{SIDE.lower()}_fullres.json'
print()
print('=' * 74); print('>>> ablation: full-res (chay lai OCR - lau)'); print('=' * 74)
!python scripts/eval_baseline.py \
    --test_jsonl '{TEST_JSONL}' --image_root '{IMAGE_ROOT}' \
    --engine {ENGINE_ABL} --image_max_side 0 \
    --report_path '{rp_full}'

>>> ablation: canonicalize (tu cache OCR)
2026-09-02 04:51:10,576 [WARNING] ⚠ ABLATION canonicalize: parser bị ép noi_cap/quoc_tich về câu mẫu theo phân bố nhãn, tức đoán trúng nhờ biết trước nhãn chứ không nhờ đọc. Đây KHÔNG phải số headline.
2026-09-02 04:51:11,535 [INFO] 📂 Đã nạp cache OCR cho 161 ảnh từ /content/drive/MyDrive/cccd_project/Data/result/back/eval_paddle_vietocr_back_ocr.jsonl
2026-09-02 04:51:11,536 [INFO] 🚀 Bắt đầu chấm điểm baseline paddle_vietocr trên tập Test...
Đánh giá: 100% 161/161 [00:09<00:00, 17.56ảnh/s]
2026-09-02 04:51:24,642 [INFO] 💾 Đã ghi 161 dự đoán → /content/drive/MyDrive/cccd_project/Data/result/back/ablation/eval_paddle_vietocr_back_canon_preds.jsonl
2026-09-02 04:51:25,502 [INFO] FA=0.7935 | CER=0.0796 | F1=0.9053 | fill=97.7% | 15.5 dòng/ảnh | p50=40.6ms
2026-09-02 04:51:25,503 [INFO] ✅ Đã lưu báo cáo → /content/drive/MyDrive/cccd_project/Data/result/back/ablation/eval_paddle_vietocr_back_canon.json

>>> ablation: full-res (chay lai OCR - lau)
20

## 8. Review the reports before moving to notebook 05

Check the three things that go wrong most often: the ablation flags are off,
`det_limit_side_len` equals `image_max_side`, and detection ran where you think it did.

In [7]:
import json

for eng in ENGINES:
    rp = RESULT_DIR / f'eval_{eng}_{SIDE.lower()}.json'
    if not rp.exists():
        print(f'{eng:16s} CHUA CO report')
        continue
    r   = json.loads(rp.read_text(encoding='utf-8'))
    run = r.get('run', {})
    cfg = r.get('engine_config') or {}

    print('=' * 74)
    print(f'{eng}   ({rp.name})')
    print('=' * 74)
    print(f'  FA                 : {r["field_accuracy"]:.2%}')
    print(f'  CER                : {r["cer"]:.4f}')
    print(f'  micro-F1           : {r["f1"]:.2%}')
    print(f'  Truong dien duoc   : {r.get("field_fill_rate", 0):.1%}   <- doc kem FA')
    print(f'  Dong OCR / anh     : {r.get("ocr_lines_per_image", "-")}')
    print(f'  mode               : {run.get("mode")}')
    print(f'  engine             : {cfg.get("engine", "-")}')
    print(f'  device (rec)       : {cfg.get("device", "-")}')
    print(f'  device (det)       : {cfg.get("det_device", cfg.get("device", "-"))}')

    ims, det = run.get('image_max_side'), run.get('det_limit_side_len')
    print(f'  image_max_side     : {ims}')
    print(f'  det_limit_side_len : {det}')
    if ims and det and ims != det:
        print('     >>> LECH: detector nhin luong pixel khac voi anh dua vao.')
    if run.get('canonicalize'):
        print('     >>> canonicalize DANG BAT - day la ABLATION, khong phai so headline.')
    if not ims:
        print('     >>> full-res DANG BAT - day la ABLATION, khong phai so headline.')
    print()

paddleocr   (eval_paddleocr_back.json)
  FA                 : 48.60%
  CER                : 0.1677
  micro-F1           : 42.15%
  Truong dien duoc   : 96.7%   <- doc kem FA
  Dong OCR / anh     : 15.2
  mode               : rule_based
  engine             : -
  device (rec)       : -
  device (det)       : -
  image_max_side     : 1024
  det_limit_side_len : 1024

paddle_vietocr   (eval_paddle_vietocr_back.json)
  FA                 : 57.14%
  CER                : 0.1092
  micro-F1           : 81.93%
  Truong dien duoc   : 97.7%   <- doc kem FA
  Dong OCR / anh     : 15.5
  mode               : rule_based
  engine             : -
  device (rec)       : -
  device (det)       : -
  image_max_side     : 1024
  det_limit_side_len : 1024



## 9. Next step

The reports are in `result/{side}/`. Open **notebook 05** and re-run it from the top — it
picks the baselines up automatically (matching `eval_*_preds.jsonl`, grouped by
`run.mode == 'rule_based'`) and adds:

- **Section 3** — bootstrap confidence interval for the baseline FA.
- **Section 4, group 3** — McNemar, fine-tuned VLM vs baseline. This is the number that
  answers L8; an FA gap alone does not support the claim.
- **Section 4, group 4** — McNemar between the two baselines, isolating the
  Vietnamese-specific recognizer.
- **Section 5** — a separate baseline table in `final_comparison_{side}.md`.

Make sure `SIDE` in notebook 05 matches this notebook.